# Delta Lake Internals

Delta Lake internals is the foundation that everything else (DLT, streaming, Unity Catalog) sits on.

We'll cover it in 4 layers, from most fundamental to most advanced:

| Layer | Topic | Why It Matters |
|---|---|---|
| 1 | Transaction Log (`_delta_log`) | The "brain" of Delta Lake: how ACID actually works |
| 2 | Time Travel | The visible feature of the transaction log |
| 3 | OPTIMIZE + ZORDER + VACUUM | File management for performance and cost |
| 4 | Checkpoints & Recovery | What happens when things break |

We'll do hands-on practice for Layers 2 and 3 (they're the most testable in Free Edition). Layers 1 and 4 are theory-heavy but crucial for interviews.

## Layer 1: The Transaction Log (`_delta_log`)

### The Simple Explanation

When you save a Delta table, Databricks doesn't just save your Parquet files. It also saves a log of every change.

```text
/my_table/
├── _delta_log/                                  ← THE BRAIN (metadata)
│   ├── 00000000000000000000.json                ← version 0 (CREATE)
│   ├── 00000000000000000001.json                ← version 1 (INSERT)
│   ├── 00000000000000000002.json                ← version 2 (UPDATE)
│   ├── 00000000000000000003.json                ← version 3 (DELETE)
│   └── 00000000000000000010.checkpoint.parquet  ← snapshot
│
├── part-00000-abc.parquet                       ← actual data files
├── part-00001-def.parquet
└── part-00002-ghi.parquet
```

### What's Inside Each JSON?

Each `00000000000000000001.json` file contains actions:

| Action | Meaning |
|---|---|
| `add` | "This Parquet file is now part of the table" |
| `remove` | "This Parquet file is no longer part of the table" |
| `metaData` | Schema, partitioning, table properties |
| `commitInfo` | Who did what, when, with what operation |

Every commit is atomic: either the whole JSON is written, or it isn't. That's how ACID works on cloud storage.

`This shows every operation ever performed on the table — with version numbers, timestamps, operations (WRITE, MERGE, OPTIMIZE), and metrics (rows added/removed).`

In [0]:
%sql
-- Create table (version 0)
CREATE OR REPLACE TABLE workspace.default.time_travel_demo (
  id INT,
  value STRING
) USING DELTA;

INSERT INTO workspace.default.time_travel_demo VALUES (1, 'original');

-- Check the version
DESCRIBE HISTORY workspace.default.time_travel_demo;

In [0]:
%sql
-- Update (new version)
UPDATE workspace.default.time_travel_demo SET value = 'updated' WHERE id = 1;

-- Insert more rows (new version)
INSERT INTO workspace.default.time_travel_demo VALUES (2, 'second row');

-- Check the version
DESCRIBE HISTORY workspace.default.time_travel_demo;

##Layer 2 : Time Travel
`Because every change is in the log, you can query any past version of your table.`

In [0]:
%sql
SELECT * FROM workspace.default.time_travel_demo;

In [0]:
%sql
-- See what it looked like at version 1 (before any changes)
SELECT * FROM workspace.default.time_travel_demo VERSION AS OF 1;

###What Kills Time Travel?
`VACUUM. It physically deletes old Parquet files. After VACUUM, time travel to versions older than the retention period (default 7 days) stops working — the underlying data is gone. `

**This is a critical interview point:** "Time travel is only as good as VACUUM allows it to be."

## Layer 3: OPTIMIZE, ZORDER, VACUUM

### The Problem: Small Files

Streaming and frequent writes create many small Parquet files. This slows down queries (file listing overhead) and increases storage cost.

### OPTIMIZE: Combine Small Files

```sql
-- Bin-pack: merge small files into larger ones (default target: ~1GB)
OPTIMIZE workspace.default.time_travel_demo;
```

- **Idempotent:** running it twice on the same data does nothing extra.
- **Doesn't change data:** it just rearranges files.
- **Readers aren't interrupted** (snapshot isolation).

### ZORDER: Co-locate Related Data

```sql
-- ZORDER: physically group rows with similar values together
OPTIMIZE workspace.default.titanic
ZORDER BY (Sex, Pclass);
```

**Why:** If you often query `WHERE Sex = 'female'`, ZORDER puts all female rows in the same files, so data skipping reads fewer files.

**Rule:** ZORDER works best for low-to-medium cardinality columns (few distinct values). Don't ZORDER on `user_id` with millions of distinct values.

### VACUUM: Delete Old Files

```sql
-- Delete files no longer referenced in the log, older than 7 days
VACUUM workspace.default.time_travel_demo;

-- See what would be deleted (dry run)
VACUUM workspace.default.time_travel_demo DRY RUN;
```

> **Warning:** VACUUM kills time travel beyond the retention window (default 7 days).


## Layer 4: Checkpoints & Recovery

### Why Checkpoints Exist

After thousands of versions, reading every JSON file from version 0 is slow. Delta Lake periodically writes a **checkpoint**: a Parquet file summarizing the entire table state at that version (by default every 10 commits).

```text
_delta_log/
├── 000...000.json                  ← version 0
├── 000...001.json                  ← version 1
├── ...
├── 000...010.checkpoint.parquet    ← snapshot at version 10
├── 000...011.json                  ← version 11
├── ...
└── _last_checkpoint                ← points readers to the latest checkpoint
```

**To reconstruct version 15:**

- If a checkpoint at version 10 exists → read checkpoint + JSONs 11–15 (fast)
- If no checkpoint → read JSONs 0–15 (slow)


####Delta Lake automatically writes checkpoints every 10 commits (versions).


## Layer 4 Practical: Checkpoints & Recovery

### Step 1: Make 15 commits

```python
spark.sql("CREATE OR REPLACE TABLE workspace.default.checkpoint_demo (id INT, val STRING)")
for i in range(1, 16):
    spark.sql(f"INSERT INTO workspace.default.checkpoint_demo VALUES ({i}, 'row_{i}')")
```

### Step 2: See the versions

```sql
DESCRIBE HISTORY workspace.default.checkpoint_demo;
```

You should see versions 0 to 15.

### Step 3: Look at the checkpoint files

```python
loc = spark.sql("DESCRIBE DETAIL workspace.default.checkpoint_demo").first()["location"]
display(dbutils.fs.ls(f"{loc}/_delta_log"))
```

Look for `00000000000000000010.checkpoint.parquet` and `_last_checkpoint`.

> **Note:** Free Edition blocks this on managed tables (`DBFS_DISABLED`, location is null). Skip it, or write the Delta table to a Volume path to see the log files.

### Step 4: Break the table

```sql
DELETE FROM workspace.default.checkpoint_demo WHERE id > 5;
UPDATE workspace.default.checkpoint_demo SET val = 'CORRUPTED';
SELECT * FROM workspace.default.checkpoint_demo;
```

Only 5 rows left, all `CORRUPTED`.

### Step 5: Check the good version

```sql
SELECT * FROM workspace.default.checkpoint_demo VERSION AS OF 15;
```

You should get 15 clean rows.

### Step 6: Recover

```sql
RESTORE TABLE workspace.default.checkpoint_demo TO VERSION AS OF 15;
SELECT COUNT(*) FROM workspace.default.checkpoint_demo;
```

Count is back to 15.

### Step 7: Confirm RESTORE is a new commit

```sql
DESCRIBE HISTORY workspace.default.checkpoint_demo;
```

The top row is `RESTORE`, and the `DELETE` and `UPDATE` are still in history.
~~~